# Week 55, Day 2: Ridge regression through the SVD

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w055/d2.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Ridge two ways

Write ridge_svd(X, y, lam), computing the ridge weights as V diag(d / (d² + λ)) Uᵀ y from the thin SVD, and shrink(X, lam), the vector of shrinkage factors. Check both against direct formulas, then print the effective degrees of freedom for several penalties.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
r = np.random.default_rng(552)
n, p = 80, 20
Z = r.normal(size=(n, p))
M = np.diag(0.85 ** np.arange(p)) @ np.linalg.qr(r.normal(size=(p, p)))[0]
X = Z @ M                                                  # a correlated design
X = X - X.mean(0)
w_true = r.normal(size=p) * 0.5
y = X @ w_true + 0.5 * r.normal(size=n); y = y - y.mean()

def ridge_svd(X, y, lam):
    U, d, Vt = np.linalg.svd(X, full_matrices=False)
    return ...

def shrink(X, lam):
    d = np.linalg.svd(X, compute_uv=False)
    return ...

d = np.linalg.svd(X, compute_uv=False)
print("singular values squared, largest to smallest:", np.round(d[[0, 5, 10, 19]] ** 2, 2))
for lam in [0.01, 1, 10, 100]:
    print(f"lambda {lam:6.2f}: effective degrees of freedom {shrink(X, lam).sum():.1f} of {p}")

In [ ]:
import numpy as np
def test_svd():
    "ridge_svd matches the direct ridge solution"
    assert np.allclose(ridge_svd(X, y, 5.0), np.linalg.solve(X.T @ X + 5.0 * np.eye(p), X.T @ y))
def test_factors():
    "shrink returns d²/(d² + λ) for each singular value"
    d = np.linalg.svd(X, compute_uv=False)
    assert np.allclose(shrink(X, 5.0), d ** 2 / (d ** 2 + 5.0))
def test_dof():
    "the effective degrees of freedom equal the trace of the ridge hat matrix"
    H = X @ np.linalg.solve(X.T @ X + 5.0 * np.eye(p), X.T)
    assert abs(shrink(X, 5.0).sum() - np.trace(H)) < 1e-8

test_svd()
test_factors()
test_dof()
print("All checks passed")

### Exercise: Same data, different units

Rescale the most influential column of X by 0.001 (as if changing its units) to make Xb. Compare least-squares and ridge (lambda = 10) predictions for X and Xb, and then ridge after standardising each column of each version.

In [ ]:
j = np.argmax(np.abs(np.linalg.lstsq(X, y, rcond=None)[0]) * X.std(0))   # the most influential regressor
Xb = X.copy(); Xb[:, j] *= 0.001
ols = lambda A: A @ np.linalg.lstsq(A, y, rcond=None)[0]
ridge = lambda A, lam=10.0: A @ ridge_svd(A, y, lam)
standardise = lambda A: (A - A.mean(0)) / A.std(0)
pred_ols_a, pred_ols_b = ols(X), ols(Xb)
pred_ridge_a, pred_ridge_b = ...
pred_std_a, pred_std_b = ...
print(f"largest change in prediction from rescaling: OLS {np.abs(pred_ols_a - pred_ols_b).max():.2e}, ridge {np.abs(pred_ridge_a - pred_ridge_b).max():.3f}, standardised ridge {np.abs(pred_std_a - pred_std_b).max():.2e}")

In [ ]:
import numpy as np
def test_ols():
    "least-squares predictions do not change when a column is rescaled"
    assert np.allclose(pred_ols_a, pred_ols_b)
def test_ridge():
    "ridge predictions do change when a column is rescaled"
    assert not np.allclose(pred_ridge_a, pred_ridge_b, atol=1e-3)
def test_std():
    "after standardising, ridge predictions are the same for both versions"
    assert np.allclose(pred_std_a, pred_std_b)

test_ols()
test_ridge()
test_std()
print("All checks passed")

### Exercise: The hat-matrix shortcut

Write loo_short(X, y, lam), the mean squared leave-one-out error from the hat-matrix shortcut, and loo_brute(X, y, lam) by refitting n times. Use the shortcut to choose lambda on a grid and compare with the penalty that minimises error on a large fresh test set.

In [ ]:
def loo_short(X, y, lam):
    H = X @ np.linalg.solve(X.T @ X + lam * np.eye(X.shape[1]), X.T)
    res = y - H @ y
    return ...

def loo_brute(X, y, lam):
    errs = []
    for i in range(len(y)):
        keep = np.arange(len(y)) != i
        w = np.linalg.solve(X[keep].T @ X[keep] + lam * np.eye(X.shape[1]), X[keep].T @ y[keep])
        errs.append(...)
    return np.mean(errs)

X_new = r.normal(size=(5000, p)) @ M
y_new = X_new @ w_true + 0.5 * r.normal(size=5000)
lams = np.logspace(-3, 3, 25)
loo = np.array([loo_short(X, y, l) for l in lams])
lam_loo = lams[np.argmin(loo)]
test_curve = np.array([np.mean((y_new - X_new @ ridge_svd(X, y, l)) ** 2) for l in lams])
lam_test = lams[np.argmin(test_curve)]
print(f"shortcut {loo_short(X, y, 3.0):.5f} against brute force {loo_brute(X, y, 3.0):.5f}")
print(f"leave-one-out chooses lambda = {lam_loo:.3f}; the test set prefers {lam_test:.3f}")
plt.semilogx(lams, loo, "o-", label="leave-one-out"); plt.semilogx(lams, test_curve, "o-", label="fresh test set"); plt.xlabel("lambda"); plt.ylabel("MSE"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_short():
    "loo_short matches brute-force leave-one-out for lambda = 3"
    assert abs(loo_short(X, y, 3.0) - loo_brute(X, y, 3.0)) < 1e-8
def test_choice():
    "the penalty chosen by leave-one-out is within a factor of 10 of the one that minimises true test error"
    assert abs(np.log10(lam_loo) - np.log10(lam_test)) <= 1

test_short()
test_choice()
print("All checks passed")